<a href="https://colab.research.google.com/github/hikari1215/wafer-defect-identification/blob/main/wafer_defect_identification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Wed Oct 28 01:16:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import os
import shutil
import random

src_dir = "/content/images"
dataset_dir = "/content/dataset_cls"
train_dir = os.path.join(dataset_dir, "train")
val_dir = os.path.join(dataset_dir, "val")

val_ratio = 0.2
random.seed(42)


if not os.path.exists(src_dir):
    print(f"Error")
else:
    class_folders = [f for f in os.listdir(src_dir) if os.path.isdir(os.path.join(src_dir, f))]
    print(f" Found {len(class_folders)} name: {class_folders}")

    for class_name in class_folders:
        class_src = os.path.join(src_dir, class_name)

        class_train = os.path.join(train_dir, class_name)
        class_val = os.path.join(val_dir, class_name)
        os.makedirs(class_train, exist_ok=True)
        os.makedirs(class_val, exist_ok=True)

        images = [f for f in os.listdir(class_src) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        random.shuffle(images)

        num_val = int(len(images) * val_ratio)
        val_imgs = images[:num_val]
        train_imgs = images[num_val:]

        for img in train_imgs:
            shutil.copy(os.path.join(class_src, img), os.path.join(class_train, img))

        for img in val_imgs:
            shutil.copy(os.path.join(class_src, img), os.path.join(class_val, img))

        print(f"   Class '{class_name}':Done {len(train_imgs)}  Train, {len(val_imgs)}  Val.")


 Found 6 name: ['class_2', 'class_5', 'class_6', 'class_1', 'class_3', 'class_4']
   Class 'class_2':Done 7  Train, 1  Val.
   Class 'class_5':Done 4  Train, 0  Val.
   Class 'class_6':Done 182  Train, 45  Val.
   Class 'class_1':Done 44  Train, 11  Val.
   Class 'class_3':Done 3207  Train, 801  Val.
   Class 'class_4':Done 232  Train, 57  Val.


In [ ]:
!pip install ultralytics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 60.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ultralytics]


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26s-cls.pt")

In [ ]:
results = model.train(data="/content/dataset_cls", epochs=50,imgsz=480,batch=32),  # run training (only run when the database is full)

Ultralytics 8.4.191 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset_cls, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=480, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=None, opset=None, optimize=False, optimizer=au